# Data Quality & Overview

First pass over a raw transaction export: load it, validate its structure,
report on its quality, and get an initial feel for the data before any
further analysis is built on top of it.

Dataset: `data/raw/finance_analytics_test_transactions.csv` — a synthetic
fixture that deliberately includes a duplicate transaction and a few invalid
rows (bad date, bad amount, missing merchant), so the checks below have
something real to catch.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

from finance_analytics.data.quality import build_quality_report
from finance_analytics.duckdb_queries import (
    connect_with_transactions,
    total_expenses,
    total_income,
    total_transaction_count,
    transactions_by_category,
    transactions_by_month,
)
from finance_analytics.io.csv import load_transactions_csv
from finance_analytics.validation.transactions import validate_transactions

pd.set_option("display.max_columns", None)

DATA_PATH = Path("../data/raw/finance_analytics_test_transactions.csv")

## 1. Load the data

`load_transactions_csv` parses `date` and `amount` explicitly. Values that
don't parse become `NaT`/`NaN` rather than a guessed, silently-wrong value —
they show up below as missing/invalid, not as bad-but-valid data.

In [2]:
transactions = load_transactions_csv(DATA_PATH)
transactions

,id,date,amount,currency,description,merchant,category,account
0,1,2026-02-02,-12.50,EUR,Morning coffee,Coffee Corner,Food & Dining,Main Account
1,2,2026-02-03,-54.90,EUR,Weekly groceries,Continente,Groceries,Main Account
2,3,2026-02-05,-29.99,EUR,Monthly subscription,Spotify,Subscriptions,Main Account
3,4,2026-02-07,-82.40,EUR,Dinner with friends,O Pescador,Food & Dining,Main Account
4,5,2026-02-10,-45.00,EUR,Electricity bill,EDP,Utilities,Main Account
5,6,2026-02-12,-18.75,EUR,Pharmacy,Farmácia Central,Health,Main Account
6,7,2026-02-15,-120.00,EUR,Train tickets,CP,Transport,Main Account
7,8,2026-02-18,-642.50,EUR,"Flight, Lisbon to Rome",TAP Air,Travel,Main Account
8,9,2026-02-20,-899.00,EUR,New laptop purchase,MediaMarkt,Shopping,Main Account
9,10,2026-02-22,-8.20,EUR,"Lunch, ""daily menu""",Café Central,Food & Dining,Main Account


## 2. Schema validation

Checks the loaded data against the expected transaction schema: required
columns, required values, and the invalid dates/amounts the loader
surfaced.

In [3]:
validation = validate_transactions(transactions)

print(f"Valid: {validation.is_valid}")
print(f"Missing columns: {list(validation.missing_columns)}")
print(f"Invalid date rows (index): {list(validation.invalid_date_rows)}")
print(f"Invalid amount rows (index): {list(validation.invalid_amount_rows)}")
print(f"Missing required values by column: {validation.missing_value_rows}")

Valid: False
Missing columns: []
Invalid date rows (index): [19]
Invalid amount rows (index): [20]
Missing required values by column: {'merchant': (21,)}


In [4]:
# Rows the validator flagged, for a closer look.
flagged_rows = sorted(
    set(validation.invalid_date_rows)
    | set(validation.invalid_amount_rows)
    | {row for rows in validation.missing_value_rows.values() for row in rows}
)
transactions.loc[flagged_rows]

,id,date,amount,currency,description,merchant,category,account
19,19,NaT,-20.0,EUR,Invalid date test,Test Merchant,Shopping,Main Account
20,20,2026-03-18,NaN,EUR,Invalid amount test,Test Merchant,Shopping,Main Account
21,21,2026-03-19,-10.0,EUR,Missing merchant test,NaN,Shopping,Main Account


## 3. Data quality report

A structural snapshot of the dataset — not an analytics report. Counts of
missing values, duplicates, and invalid rows, plus a few descriptive facts
(unique merchants/categories, date range, income vs. expense counts).

In [5]:
quality_report = build_quality_report(transactions)
quality_report.to_frame()

,value
Rows,22
Columns,8
Duplicate rows,1
Duplicate transaction IDs,1
Invalid dates,1
Invalid amounts,1
Unique merchants,16
Unique categories,10
Date range,2026-02-02 to 2026-03-19
Income transactions,2


## 4. Descriptive statistics

Basic distribution of transaction amounts (invalid amounts are `NaN` and are
excluded automatically).

In [6]:
transactions["amount"].describe()

count      21.000000
mean       81.394762
std       814.871953
min      -899.000000
25%       -61.300000
50%       -29.990000
75%       -15.990000
max      3450.000000
Name: amount, dtype: float64

## 5. Date range

In [7]:
valid_dates = transactions["date"].dropna()
print(f"Earliest transaction: {valid_dates.min().date()}")
print(f"Latest transaction:   {valid_dates.max().date()}")
print(f"Span:                 {(valid_dates.max() - valid_dates.min()).days} days")

Earliest transaction: 2026-02-02
Latest transaction:   2026-03-19
Span:                 45 days


## 6. Category distribution

In [8]:
transactions["category"].value_counts()

category
Food & Dining    4
Subscriptions    4
Shopping         4
Groceries        2
Transport        2
Income           2
Utilities        1
Health           1
Travel           1
Cash             1
Name: count, dtype: int64

## 7. Income vs. expenses

Using the sign convention from the source data: positive amounts are
income, negative amounts are expenses. Rows with an invalid amount are
excluded from both.

In [9]:
valid_amounts = transactions["amount"].dropna()

pd.DataFrame(
    {
        "count": [int((valid_amounts > 0).sum()), int((valid_amounts < 0).sum())],
        "total": [valid_amounts[valid_amounts > 0].sum(), valid_amounts[valid_amounts < 0].sum()],
    },
    index=["income", "expenses"],
)

,count,total
income,2,3950.00
expenses,19,-2240.71


## 8. DuckDB queries

The same DataFrame, queryable with SQL. Useful once questions get more
relational (grouping, filtering, joins across periods) than plain pandas
makes comfortable.

In [10]:
con = connect_with_transactions(transactions)

print(f"Total transactions: {total_transaction_count(con)}")
print(f"Total income:       {total_income(con):.2f}")
print(f"Total expenses:     {total_expenses(con):.2f}")

Total transactions: 22
Total income:       3950.00
Total expenses:     -2240.71


In [11]:
transactions_by_category(con)

,category,transaction_count,total_amount
0,Food & Dining,4,-140.90
1,Subscriptions,4,-105.96
2,Shopping,4,-929.00
3,Transport,2,-142.40
4,Income,2,3950.00
5,Groceries,2,-116.20
6,Health,1,-18.75
7,Utilities,1,-45.00
8,Cash,1,-100.00
9,Travel,1,-642.50


In [12]:
transactions_by_month(con)

,month,transaction_count,total_amount
0,2026-02,12,1520.77
1,2026-03,9,208.52


## 9. Visualisation

Spending by category (expenses only — the `Income` category is a different
kind of thing and would distort a "where did the money go" chart).

In [13]:
spending_by_category = transactions_by_category(con)
spending_by_category = spending_by_category[spending_by_category["total_amount"] < 0].copy()
spending_by_category["spend"] = spending_by_category["total_amount"].abs()
spending_by_category = spending_by_category.sort_values("spend", ascending=True)

fig = px.bar(
    spending_by_category,
    x="spend",
    y="category",
    orientation="h",
    title="Spending by Category",
    labels={"spend": "Total spent", "category": ""},
)
fig.update_layout(showlegend=False)
fig.show()